# One-Class SVM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

## Load data and split

In [ ]:
data = pd.read_csv('../data/labeled.csv', parse_dates=['timestamp'])
split_point = int(len(data) * 0.8)
train_data = data.iloc[:split_point]
test_data = data.iloc[split_point:]

feature_columns = ['memory_pct', 'roll_mean_1h', 'roll_std_1h', 'roll_mean_24h',
                   'diff_1', 'diff_6', 'hour', 'minute', 'time_of_day', 'day_of_week', 'is_weekend']

X_train = train_data[feature_columns].values
X_test = test_data[feature_columns].values
y_test = test_data['label'].values

# Scaling is required for OCSVM with RBF kernel
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Anomalies in test:', y_test.sum())

## Train One-Class SVM

In [ ]:
ocsvm_model = OneClassSVM(kernel='rbf', nu=0.005, gamma='scale')
ocsvm_model.fit(X_train_scaled)

ocsvm_predictions = (ocsvm_model.predict(X_test_scaled) == -1).astype(int)
ocsvm_scores = -ocsvm_model.score_samples(X_test_scaled)

print('Flagged points:', ocsvm_predictions.sum())

## Plot detections

In [ ]:
timestamps = test_data['timestamp'].values
memory = test_data['memory_pct'].values

plt.figure(figsize=(14, 4))
plt.plot(timestamps, memory, color='lightblue')
plt.scatter(timestamps[y_test == 1], memory[y_test == 1], color='red', marker='x', label='True anomaly')
plt.scatter(timestamps[ocsvm_predictions == 1], memory[ocsvm_predictions == 1], color='orange', s=10, label='OCSVM flagged')
plt.title('One-Class SVM')
plt.ylabel('Memory %')
plt.legend()
plt.show()

## Evaluation

In [ ]:
print(classification_report(y_test, ocsvm_predictions, target_names=['Normal','Anomaly'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, ocsvm_predictions))
print('ROC-AUC:', round(roc_auc_score(y_test, ocsvm_scores), 4))